In [ ]:
import pandas as pd

# --- 1. 配置区 ---
data_dir = "../data/资产配置/处理后数据"
target_code = ("000007")
start_date_str = "2023-03-01"
end_date_str = "2025-12-31"

# CSV 文件名都带有 2303-2512 后缀
static_file = '基金主要信息2303-2512.csv'
timeseries_files = [
    '资产配置2303-2512.csv',
    '股票配置2303-2512.csv',
    '按行业股票配置表2303-2512.csv',
    '债券配置2303-2512.csv',
]

try:
    # --- 2. 提取静态信息 ---
    csv_static_path = f"{data_dir}/{static_file}"
    profile_all = pd.read_csv(csv_static_path, encoding='utf-8-sig', low_memory=False)
    code_col = next((c for c in profile_all.columns if c.upper() == 'MASTERFUNDCODE'), 'MasterFundCode')
    profile_df = profile_all[profile_all[code_col].astype(str).str.zfill(6) == target_code].copy()

    # --- 3. 提取动态时间序列 ---
    ts_dfs = []
    for csv_file in timeseries_files:
        csv_path = f"{data_dir}/{csv_file}"
        df = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)

        # 自动检测列名
        cols = df.columns.tolist()
        date_col = next((c for c in cols if c.upper() == 'ENDDATE'), 'EndDate')
        code_col = next((c for c in cols if c.upper() == 'MASTERFUNDCODE'), 'MasterFundCode')

        # 统一字段名
        df = df.rename(columns={code_col: 'MasterFundCode', date_col: 'EndDate'})

        # 按基金代码筛选
        df = df[df['MasterFundCode'].astype(str).str.zfill(6) == target_code]

        if not df.empty:
            # 强制转换 EndDate 为 Pandas 时间对象
            df['EndDate'] = pd.to_datetime(df['EndDate'], errors='coerce')

            # 按日期范围筛选
            mask = (df['EndDate'] >= start_date_str) & (df['EndDate'] <= end_date_str)
            df = df.loc[mask].copy()

            if not df.empty:
                # 转换回字符串显示格式
                df['EndDate'] = df['EndDate'].dt.strftime('%Y-%m-%d')

                # 清洗代码格式
                df['MasterFundCode'] = df['MasterFundCode'].apply(lambda x: str(int(float(x))).zfill(6) if pd.notnull(x) else "")

                # 处理明细序号
                df['Seq'] = df.groupby(['MasterFundCode', 'EndDate']).cumcount() + 1
                df = df.set_index(['MasterFundCode', 'EndDate', 'Seq'])

                # 加前缀
                df.columns = [f"{csv_file.replace('.csv','')}_{col}" for col in df.columns]
                ts_dfs.append(df)
                print(f"  - 成功提取并过滤数据: {csv_file}")

    # --- 4. 生成报告 ---
    output_file = f"Fund_Full_Report_{target_code}.md"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"# 基金分析报告: {target_code} (2023-2025)\n\n")
        f.write("## [附注: 基金基础信息]\n")
        f.write(profile_df.to_markdown(index=False) if not profile_df.empty else "无数据")

        f.write("\n\n---\n\n")
        f.write("## [历史配置明细数据]\n")

        if ts_dfs:
            final_ts_df = pd.concat(ts_dfs, axis=1, join='outer').reset_index()
            # 确保排序正确
            final_ts_df = final_ts_df.sort_values(by=['EndDate', 'Seq'], ascending=[False, True])
            f.write(final_ts_df.fillna("-").to_markdown(index=False))
        else:
            f.write("未能提取到有效的时间序列数据，请检查 CSV 文件路径。")

    print(f"\n新报告已生成: {output_file}")

except Exception as e:
    print(f"执行失败: {e}")